In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Compute the 2D Discrete Fourier Transform (2D DFT) of a complex-valued signal stored on the GPU.
  Given a 2D complex input signal of shape <code>M &times; N</code>, compute its 2D DFT spectrum
  using the row-column decomposition: apply a 1D DFT along each row, then a 1D DFT along each
  column of the result. All values are 32-bit floating point.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in <code>spectrum</code></li>
  <li>
    The input and output are stored as 1D arrays of interleaved real and imaginary parts in
    row-major order: element <code>x[m, n]</code> has its real part at index
    <code>2*(m*N + n)</code> and imaginary part at index <code>2*(m*N + n) + 1</code>
  </li>
</ul>

<h2>Example</h2>
<p>
Input: <code>M</code> = 2, <code>N</code> = 2<br>
Signal $x[m, n]$ (real part):
$$
\begin{bmatrix}
1.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
Signal $x[m, n]$ (imaginary part):
$$
\begin{bmatrix}
0.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
Output:<br>
Spectrum $X[k, l]$ (real part):
$$
\begin{bmatrix}
1.0 & 1.0 \\
1.0 & 1.0
\end{bmatrix}
$$
Spectrum $X[k, l]$ (imaginary part):
$$
\begin{bmatrix}
0.0 & 0.0 \\
0.0 & 0.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>M</code>, <code>N</code> &le; 4096</li>
  <li>Signal values are 32-bit floating point (real and imaginary parts)</li>
  <li>Performance is measured with <code>M</code> = 2,048, <code>N</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// signal, spectrum are device pointers
extern "C" void solve(const float* signal, float* spectrum, int M, int N) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# signal, spectrum are tensors on the GPU
@cute.jit
def solve(signal: cute.Tensor, spectrum: cute.Tensor, M: cute.Int32, N: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# signal is a tensor on GPU
@jax.jit
def solve(signal: jax.Array, M: int, N: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


# signal, spectrum are device pointers
@export
def solve(
    signal: UnsafePointer[Float32, MutExternalOrigin],
    spectrum: UnsafePointer[Float32, MutExternalOrigin],
    M: Int32,
    N: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# signal, spectrum are tensors on the GPU
def solve(signal: torch.Tensor, spectrum: torch.Tensor, M: int, N: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# signal, spectrum are tensors on the GPU
def solve(signal: torch.Tensor, spectrum: torch.Tensor, M: int, N: int):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/78_2d_fft/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
